# バッチ処理と GPU で推論を高速化する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/08_transformers/batch_inference.ipynb)

NLP クックブック 08-06 ／ Transformers の基本操作

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

数万件のテキストに対してモデルの推論を回したい。1 件ずつ処理すると遅すぎる。

In [1]:
import time
from statistics import median
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
name = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name).to(device).eval()

base = ["とても良かった。", "最悪だった。二度と行かない。", "普通でした。可もなく不可もなく、値段相応という印象です。" * 4]
texts = [base[2] if i % 8 == 0 else base[i % 2] for i in range(96)]   # 8 件に 1 件だけ長い文

@torch.inference_mode()
def predict(texts, batch_size, sort_by_length=False):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i])) if sort_by_length else list(range(len(texts)))
    padding = 0
    preds = [None] * len(texts)
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        batch = tokenizer([texts[j] for j in idx], padding=True, truncation=True, max_length=256, return_tensors="pt")
        padding += (batch.attention_mask == 0).sum().item()
        batch = batch.to(device)
        for j, p in zip(idx, model(**batch).logits.argmax(-1).tolist()):
            preds[j] = model.config.id2label[p]
    return preds, padding

print("device:", device)
reference, _ = predict(texts, 1)
rows = []
for label, size, sort in [("1 件ずつ", 1, False), ("バッチ 32", 32, False), ("バッチ 32 + 長さでソート", 32, True)]:
    predict(texts, size, sort)  # 各条件でウォームアップ
    seconds = []
    for _ in range(3):
        if device == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        preds, padding = predict(texts, size, sort)
        if device == "cuda":
            torch.cuda.synchronize()
        seconds.append(time.perf_counter() - start)
    rows.append({"条件": label, "秒（中央値）": round(median(seconds), 3),
                 "パディング数": padding, "予測一致数": sum(a == b for a, b in zip(preds, reference)),
                 "全件数": len(texts)})
print(pd.DataFrame(rows).to_string(index=False))

device: cuda
             条件  秒（中央値）  パディング数  予測一致数  全件数
          1 件ずつ   0.419       0     96   96
         バッチ 32   0.230    7020     96   96
バッチ 32 + 長さでソート   0.101    1644     96   96


- `torch.inference_mode()`（または `no_grad()`）で勾配計算を止める。忘れるとメモリを浪費し、遅くなる。
- 長さの近い文を同じバッチにまとめると、パディングが減って速くなる。結果は元の順番に戻す。
- 予測一致数を全件数と比べ、元の順番で同じラベルになったか確かめる。時間はトークン化・転送・予測を含む。バッチ32の2行で、パディング数と時間の変化を比べる。
- CPU では、長さを揃えずにバッチにすると、長い文に合わせたパディングの計算が重く、1 件ずつより遅くなることがある。
- GPU の処理時間を測るときは、最初の 1 回（初期化やカーネルの準備を含む）を計測から外す。
- `pipeline(..., device=0, batch_size=32)` でも同様のバッチ処理ができる。